# 🌾 EDA – Barley Production (France)

In [ ]:
import sys, os

# Add project root to sys.path so 'constants' is importable
PROJECT_ROOT = "/Users/gregzguegue/Desktop/DSB/24_Data_for_Strat_BCG/BCG-Data-for-Strategy"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from constants.path import CLIMATE_PATH, BARLEY_PATH, GOLD_PATH, SILVER_PATH
from constants.constants import CLIMATE_COLUMNS, DEPARTMENT_COLUMNS

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


In [ ]:
df = pd.read_parquet(SILVER_PATH / "barley.parquet")
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

---
## 1 · Analyse temporelle globale
Évolution annuelle moyenne du **rendement**, de la **production** et de la **surface cultivée**.

In [ ]:
yearly = df.groupby("year")[["yield", "production", "area"]].mean()


fig, ax1 = plt.subplots(figsize=(14, 6))


color_yield = "#2563EB"

color_prod = "#16A34A"

color_area = "#DC2626"


ax1.set_xlabel("Year")

ax1.set_ylabel("Average Yield (t/ha)", color=color_yield)

ax1.plot(yearly.index, yearly["yield"], color=color_yield, linewidth=2.2,

marker="o", markersize=4, label="Yield (t/ha)")

ax1.tick_params(axis="y", labelcolor=color_yield)


ax2 = ax1.twinx()

ax2.set_ylabel("Production / Cultivated Area", color=color_prod)

ax2.plot(yearly.index, yearly["production"], color=color_prod, linewidth=2,

linestyle="--", label="Average Production (t)")

ax2.plot(yearly.index, yearly["area"], color=color_area, linewidth=2,

linestyle=":", label="Average Cultivated Area (ha)")

ax2.tick_params(axis="y", labelcolor=color_prod)


lines1, labels1 = ax1.get_legend_handles_labels()

lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left",

frameon=True, fancybox=True, shadow=True)


ax1.set_title("Average Yearly Evolution – Yield, Production, Cultivated Area",
              fontweight="bold")

fig.tight_layout()

plt.show() 

In [ ]:
yearly_area = df.groupby("year")[["area"]].mean()

# Calculate the 10-year percentage variation
yearly_area["decadal_var"] = yearly_area["area"].pct_change(periods=10) * 100

fig, ax1 = plt.subplots(figsize=(14, 7))

# Primary Plot
ax1.plot(yearly_area.index, yearly_area["area"], color="#DC2626", linewidth=3, 
         marker='o', markersize=5, label="Cultivated Area (ha)", zorder=3)

# Start and end years
start_year = yearly_area.index.min()
end_year = yearly_area.index.max()

# Clearer Decadal Highlighting
for year in range(start_year, end_year + 1):
    # Add a vertical line and text every 10 years starting from the first data point
    if (year - start_year) % 10 == 0 and year != start_year:
        ax1.axvline(x=year, color='gray', linestyle='--', alpha=0.5, zorder=1)
        
        # Get the variation value
        if year in yearly_area.index and not pd.isna(yearly_area.loc[year, "decadal_var"]):
            var_val = yearly_area.loc[year, "decadal_var"]
            color_var = "#16A34A" if var_val >= 0 else "#DC2626" # Green if growth, Red if loss
            
            ax1.annotate(f'10Y Var: {var_val:.1f}%', 
                         xy=(year, yearly_area.loc[year, "area"]), 
                         xytext=(0, 15), 
                         textcoords='offset points', 
                         ha='center', 
                         fontsize=10, 
                         fontweight='bold',
                         color='white',
                         bbox=dict(boxstyle='round,pad=0.5', fc=color_var, ec='none', alpha=0.9))

# Highlighting the starting point
ax1.scatter(start_year, yearly_area.loc[start_year, "area"], color='black', s=100, label="Reference Start", zorder=4)

# Formatting
ax1.set_xlabel("Year")
ax1.set_ylabel("Average Cultivated Area (ha)")
ax1.set_title("Evolution of Cultivated Area with Decadal Growth Markers", fontsize=14, fontweight="bold")
ax1.grid(True, axis='y', linestyle=':', alpha=0.6)
ax1.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

yearly_area = df.groupby("year")["area"].mean().sort_index()

# Calculate specific variations
first_year = yearly_area.index[0]
last_year = yearly_area.index[-1]
area_start = yearly_area.iloc[0]
area_end = yearly_area.iloc[-1]

# Overall variation
total_var = ((area_end - area_start) / area_start) * 100

# Find values closest to 10 and 5 years ago
idx_10y = yearly_area.index.get_indexer([last_year - 10], method='nearest')[0]
idx_5y = yearly_area.index.get_indexer([last_year - 5], method='nearest')[0]

year_10y = yearly_area.index[idx_10y]
year_5y = yearly_area.index[idx_5y]

var_10y = ((area_end - yearly_area.iloc[idx_10y]) / yearly_area.iloc[idx_10y]) * 100
var_5y = ((area_end - yearly_area.iloc[idx_5y]) / yearly_area.iloc[idx_5y]) * 100

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_area.index, yearly_area.values, color="#DC2626", lw=2, marker='o', markersize=4, label="Mean Area")

# Summary text box
stats_text = (
    f"Summary of Variations:\n"
    f"Total ({first_year}-{last_year}): {total_var:+.1f}%\n"
    f"Past 10 Years ({year_10y}-{last_year}): {var_10y:+.1f}%\n"
    f"Past 5 Years ({year_5y}-{last_year}): {var_5y:+.1f}%"
)

ax.text(0.02, 0.95, stats_text, transform=ax.transAxes, verticalalignment='top', 
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9), fontsize=11)

# Formatting
ax.set_xlabel("Year")
ax.set_ylabel("Average Cultivated Area (ha)")
ax.set_title("Cultivated Area Variation Analysis", fontsize=14)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
yearly_area = df.groupby("year")["area"].mean().sort_index()

In [ ]:
yearly_area

In [ ]:
yearly = df.groupby("year")[["yield", "production", "area"]].mean()

fig, ax1 = plt.subplots(figsize=(14, 6))

# Colors for the metrics
color_yield = "#2563EB"
color_area  = "#DC2626"

# Primary Axis: Yield
ax1.set_xlabel("Year")
ax1.set_ylabel("Average Yield (t/ha)", color=color_yield)
ax1.plot(yearly.index, yearly["yield"], color=color_yield, linewidth=2.2,
         marker="o", markersize=4, label="Yield (t/ha)")
ax1.tick_params(axis="y", labelcolor=color_yield)

# Secondary Axis: Area
ax2 = ax1.twinx()
ax2.set_ylabel("Average Cultivated Area (ha)", color=color_area)
ax2.plot(yearly.index, yearly["area"], color=color_area, linewidth=2,
         linestyle="--", label="Average Area (ha)")
ax2.tick_params(axis="y", labelcolor=color_area)

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left",
           frameon=True, fancybox=True, shadow=True)

# English Title
ax1.set_title("Average Yearly Evolution – Yield vs. Cultivated Area",
              fontweight="bold")

fig.tight_layout()
plt.show()

---
## 2 · Winners & Loosers
Taux de croissance du rendement entre la première et la dernière année pour chaque département.

In [ ]:
# Rendement moyen par département à la première et dernière année disponibles
first_year = df["year"].min()
last_year  = df["year"].max()
print(f"Période analysée : {first_year} → {last_year}")

yield_first = (
    df[df["year"] == first_year]
    .groupby("department")["yield"].mean()
    .rename("yield_first")
)
yield_last = (
    df[df["year"] == last_year]
    .groupby("department")["yield"].mean()
    .rename("yield_last")
)

growth = pd.concat([yield_first, yield_last], axis=1).dropna()
growth["growth_rate"] = (
    (growth["yield_last"] - growth["yield_first"]) / growth["yield_first"]
) * 100

growth = growth.sort_values("growth_rate", ascending=False)

winners = growth.head(3)
loosers = growth.tail(3)

print("\n🏆  TOP 3 Winners (meilleure progression) :")
print(winners[["growth_rate"]].to_string())
print("\n📉  TOP 3 Loosers (plus forte baisse) :")
print(loosers[["growth_rate"]].to_string())


In [ ]:
# Courbes comparatives Winners vs Loosers
focus_deps = list(winners.index) + list(loosers.index)
df_focus = df[df["department"].isin(focus_deps)]

palette = {dep: color for dep, color in zip(
    focus_deps,
    ["#2563EB", "#3B82F6", "#93C5FD",   # bleus → winners
     "#EF4444", "#DC2626", "#B91C1C"],   # rouges → loosers
)}

fig, ax = plt.subplots(figsize=(14, 7))
for dep in focus_deps:
    sub = df_focus[df_focus["department"] == dep].sort_values("year")
    label_suffix = "🏆" if dep in winners.index else "📉"
    ax.plot(sub["year"], sub["yield"], linewidth=2, marker="o", markersize=3,
            color=palette[dep], label=f"{dep} {label_suffix}")

ax.set_title("Rendement – Winners vs Loosers", fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Rendement (t/ha)")
ax.legend(loc="upper left", frameon=True, fancybox=True, shadow=True)
fig.tight_layout()
plt.show()


---
## 3 · Leaders – Top absolu
Département avec la **production moyenne** la plus haute et celui avec le **rendement moyen** le plus haut.

In [ ]:
dep_stats = df.groupby("department").agg(
    avg_production=("production", "mean"),
    avg_yield=("yield", "mean"),
    avg_area=("area", "mean"),
).sort_values("avg_production", ascending=False)

top_producer = dep_stats["avg_production"].idxmax()
top_yielder  = dep_stats["avg_yield"].idxmax()

print(f"🏭  Plus gros producteur (production moy.) : {top_producer}"
      f"  →  {dep_stats.loc[top_producer, 'avg_production']:,.0f} t")
print(f"🎯  Plus efficace (rendement moy.)          : {top_yielder}"
      f"  →  {dep_stats.loc[top_yielder, 'avg_yield']:.2f} t/ha")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 par production
top10_prod = dep_stats.nlargest(10, "avg_production")
axes[0].barh(top10_prod.index[::-1], top10_prod["avg_production"][::-1],
             color="#2563EB", edgecolor="white")
axes[0].set_title("Top 10 – Production moyenne (t)", fontweight="bold")
axes[0].set_xlabel("Production moyenne (t)")

# Top 10 par rendement
top10_yield = dep_stats.nlargest(10, "avg_yield")
axes[1].barh(top10_yield.index[::-1], top10_yield["avg_yield"][::-1],
             color="#16A34A", edgecolor="white")
axes[1].set_title("Top 10 – Rendement moyen (t/ha)", fontweight="bold")
axes[1].set_xlabel("Rendement moyen (t/ha)")

fig.tight_layout()
plt.show()


---
## 4 · Focus stratégique – Zones ClientCo
Départements : **Essonne, Somme, Cher, Haute-Garonne, Isère**.

In [ ]:
CLIENTCO_DEPS = ["Essonne", "Somme", "Cher", "Haute-Garonne", "Isère"]
df_clientco = df[df["department"].isin(CLIENTCO_DEPS)].copy()

print(f"Nombre de lignes pour les zones ClientCo : {len(df_clientco)}")
df_clientco.head()


In [ ]:
# Bar chart groupé – rendement annuel par département ClientCo
pivot = df_clientco.pivot_table(index="year", columns="department",
                                values="yield", aggfunc="mean")

colors = ["#2563EB", "#16A34A", "#F59E0B", "#DC2626", "#8B5CF6"]
ax = pivot.plot(kind="bar", figsize=(18, 6), width=0.75, color=colors,
                edgecolor="white", linewidth=0.5)
ax.set_title("Rendement annuel – Zones ClientCo", fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Rendement (t/ha)")
ax.legend(title="Département", loc="upper left", frameon=True,
          fancybox=True, shadow=True)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plot – Surface vs Production, taille = rendement
avg_clientco = df_clientco.groupby("department").agg(
    avg_area=("area", "mean"),
    avg_production=("production", "mean"),
    avg_yield=("yield", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    avg_clientco["avg_area"],
    avg_clientco["avg_production"],
    s=avg_clientco["avg_yield"] * 80,   # taille proportionnelle au rendement
    c=colors[:len(avg_clientco)],
    alpha=0.85,
    edgecolors="white",
    linewidth=1.5,
)

for _, row in avg_clientco.iterrows():
    ax.annotate(
        row["department"],
        (row["avg_area"], row["avg_production"]),
        textcoords="offset points", xytext=(10, 5),
        fontsize=11, fontweight="bold",
    )

ax.set_title("Surface vs Production – Zones ClientCo\n"
             "(taille des points ∝ rendement)", fontweight="bold")
ax.set_xlabel("Surface moyenne (ha)")
ax.set_ylabel("Production moyenne (t)")
fig.tight_layout()
plt.show()
